1. Executive Summary

This report investigates the implementation of the rotation family $R_z(\pi/2^k)$ within a fault-tolerant framework. While 1-qubit rotations appear mathematically trivial compared to 2-qubit entangling gates, this research demonstrates that at the logical level, 1-qubit non-Clifford gates often impose significantly higher resource costs. Through circuit synthesis and simulation using the Bloqade ecosystem, we show that small-angle rotations require extensive decomposition and indirect "injection" protocols that exceed the complexity of standard 2-qubit Clifford operations.

In [1]:
from typing import Any

from kirin.ir import Method
import matplotlib.pyplot as plt
import numpy as np

from bloqade import squin, tsim
from bloqade.cirq_utils import emit_circuit, load_circuit, noise
from bloqade.pyqrack import StackMemorySimulator
from bloqade.types import MeasurementResult, Qubit
from kirin.dialects.ilist import IList
import matplotlib.pyplot as plt

# this will help us have return types for our methods that have more intuitive names
Register = IList[Qubit, Any]
Measurement = IList[MeasurementResult, Any]

# this function will help us visualize some circuits
def show_circuit(squin_kernel):
    @squin.kernel
    def _to_visualize():
        _ = squin_kernel()

    return tsim.Circuit(_to_visualize).diagram(height=400)

An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


We first tried to familiarize with the family of the Clifford +T gates

In [32]:
@squin.kernel
def circuit() -> Measurement:
    q = squin.qalloc(2) 
    
    squin.t(q[0])
    squin.h(q[0])
    squin.s(q[0])
    squin.cx(q[0],q[1])

    b= squin.broadcast.measure(q)

    return q

show_circuit(circuit)


In [2]:
%pip install pygridsynth

/home/sytrams/.local/share/uv/python/cpython-3.12.13-linux-x86_64-gnu/lib/python3.12/pty.py:95: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  pid, fd = os.forkpty()


Note: you may need to restart the kernel to use updated packages.


2. Implementation and Results

We explored the approximation of $R_z(\pi/2^k)$ for $k \in \{0, \dots, 5\}$.Algorithmic Decomposition: Using the Ross and Selinger algorithm, target rotations were translated into strings of $H$ and $T$ gates. 

In [13]:
import mpmath

from pygridsynth.gridsynth import gridsynth_gates

def clifford_string (i:int)-> str:
    mpmath.mp.dps = 128
    theta = mpmath.mp.pi / (2**i)
    epsilon = mpmath.mpf("1e-10")
    gates = gridsynth_gates(theta=theta, epsilon=epsilon)
    #print(gates)
    return gates

Precision vs. Cost: Our simulations confirmed that as the angle $\theta$ decreases (increasing $k$), the number of required $T$ gates grows logarithmically. Convergence Limits: For $k \geq 7$, the synthesis depth required to maintain a distance metric $d < 10^{-7}$ becomes computationally impractical for standard simulation, marking a "complexity wall" in 1-qubit synthesis.

In [ ]:
import math
from functools import reduce

H = (1/math.sqrt(2)) * np.array([[1.0, 1.0],[1.0, -1.0]], dtype=np.complex128)
S = np.array([[1.0 ,0.0],[0.0, 1j]], dtype=np.complex128)
T = np.array([[1, 0], [0, np.exp(1j * np.pi / 4)]], dtype=np.complex128)
I=np.array([[1.0, 1.0],[1.0,1.0]], dtype=np.complex128)


def Rz(theta: float) -> np.ndarray:
    """Gate di rotazione sull'asse Z di angolo theta."""
    return np.array([
        [np.exp(-1j * theta / 2), 0],
        [0, np.exp(1j * theta / 2)]
    ])

def d(U: np.ndarray, V: np.ndarray) -> float:
    U_dag = U.conj().T          
    trace = np.trace(U_dag @ V) 
    result = 1 - abs(trace) / 2 
    return np.sqrt(result)

k=3

map_matrix = {
    'H' : H,
    'T' : T,
    'S' : S
}
while True:
    theta = np.pi/2**k
    rot = Rz(theta)

    input_string = clifford_string (k)

    matrix_list = [map_matrix[c] for c in input_string if c in map_matrix]

    U = np.array(reduce(lambda x, y: x @ y, matrix_list), dtype=np.complex128)

    dist = d(U,rot)

    dist_null = d(I,rot)

    if dist>dist_null :
        print (f"K={k} -> It's not worth it, go to sleep")
        break;
    if k>10 :
        print ("You've done enough, go to sleep")
        break;

    print (f"K={k} -> distance = {d(U,rot):.12f}")

    k=k+1


K=3 -> distance = 0.000000094830
K=4 -> distance = 0.000000098843
K=5 -> distance = 0.000000097713
K=6 -> distance = 0.000000099403
K=7 -> It's not worth it, go to sleep


3. Gate Injection and Resource Overhead

In settings where $T$ gates cannot be applied directly to a data qubit, we implemented a Gate Injection protocol. Protocol Structure: A "Magic State" $|T\rangle$ is prepared on an auxiliary qubit, followed by a $CNOT$ gate between the data and auxiliary qubits. Feed-forward Requirements: The implementation requires a measurement of the auxiliary qubit and a conditional $S^\dagger$ (S-adjoint) correction. Findings: A single 1-qubit $T$ rotation effectively requires one auxiliary qubit, one 2-qubit gate, and active classical feed-forward logic.

In [ ]:
for i in range (3):
    input = clifford_string (3+i)

    @squin.kernel
    def t_injection () -> Register:
        q = squin.qalloc (2)

        squin.h(q[1])
        squin.t(q[1])

        for char in input:
            if char=="H":
                squin.h(q[0])
            if char=="S":
                squin.s(q[0])
            if char=="T":
                squin.cx(q[0], q[1])
                if squin.measure(q[1])==False:
                    squin.s_adj(q[0])
                else :
                    squin.reset(q[1])
        return q
    

Here's an example of a circuit implementation.

In [ ]:
@squin.kernel
def circuit6() -> Register:
    q = squin.qalloc(2) 
    
    squin.s(q[0])
    squin.h(q[0])
    squin.s(q[0])

    squin.h(q[1])
    squin.t(q[1])
    

    
    squin.cx(q[0], q[1])
    bits= squin.measure (q[1])
    squin.reset(q[1])
    
    
    squin.s_adj(q[0])


    squin.h(q[0])
    squin.s(q[0])
    squin.h(q[0])

    squin.h(q[1])
    squin.t(q[1])

    squin.cx(q[0], q[1])
    bits= squin.measure (q[1])
     
    return q

show_circuit(circuit6)

4. A step further

After successfully implementing the T-gate injection protocol, a natural next step for technical verification is to ensure the conditional correction ($S^\dagger$) was applied accurately based on the measurement outcome. This can be achieved by performing a basis-change verification to check the consistency of the final state.

Once the $T$ gate is "injected" into the data qubit and the feed-forward correction is complete, the qubit should theoretically reside in the targeted state. To verify this, one can apply an "opposite" gate—essentially the inverse of the expected logical operation—and then measure in the X-basis rather than the standard computational (Z) basis.

If the injection and $S^\dagger$ correction were successful, the state should be deterministic when rotated into the X-basis.

5. Conclusion

The experimental results support the counter-intuitive claim: A 1-qubit gate can be harder to implement than a 2-qubit gate. In a fault-tolerant environment, 2-qubit Clifford gates (like $CNOT$) are relatively "cheap" due to their transversal nature in many codes. Conversely, a "simple" 1-qubit rotation requires:

    -Extensive synthesis into discrete gate sequences.
    -Ancilla qubit preparation and consumption.
    -Integration of 2-qubit gates and measurement-based corrections
    
This complexity is the primary bottleneck in scaling algorithms like the Quantum Fourier Transform (QFT) to the logical level.